<h1>Setting up RAG<h1>

In [ ]:
#5. RAG
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

# 1) Load the same embedding model you used when indexing
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# 2) Load the persisted vector store
vectorstore = Chroma(
    persist_directory="./chroma_langchain_db",
    collection_name="uia_courses",
    embedding_function=embeddings,
)

# 3) Turn it into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 4) Load your Ollama LLM
llm = ChatOllama(
    model="qwen2.5:0.5b",   # or another model you actually pulled
    temperature=0,
    base_url="http://localhost:11434"
)
print("Loaded RAG system")

<h1>Example question<h1>

In [ ]:

# 5)
question = "What are the learning outcomes of the deep neural network course?"

# Retrieve relevant chunks
docs = retriever.invoke(question)

# Build context
context = "\n\n".join(doc.page_content for doc in docs)

# Prompt the LLM
prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say you do not know.

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print("QUESTION:")
print(question)
print("\nRETRIEVED DOCUMENTS:")
for i, doc in enumerate(docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)

print("\nANSWER:")
print(response.content)